# KBU 실내환경 모니터링 v4
온톨로지 변환 → 규칙 판단 → 트렌드 예측 → LLM 해석 → 에어컨 제어

**사용 데이터 및 근거**:
- `telemetry-cal/` : EDC 보정값 — raw와 co2/temp/hum 동일, aerosol/gas는 보정값 우선
- `telemetry-agg/` : EDC 1h 집계 — 트렌드 예측 (30s/1M은 예외 케이스, 제외)
- `radar-frame/`   : RDC 레이더 — targetCount(재실), t1~t3(위치 기반 가중치)
- `thermal-frame/` : IRC 열화상 통계 — stats_min/max/avg/center

**제외 데이터 및 근거**:
- `telemetry/` (raw): cal과 co2/temp/hum 동일, aerosol/gas는 cal 보정값 우선
- `thermal-pixels/` : 30GB .csv.gz, 실시간 처리 불가, thermal-frame 통계로 충분
- EDC-KBU-23~27: 2026-03-31 단 1일치 데이터만 존재
- RDC-KBU-02: 폴더 내 데이터 파일 없음 (tree 확인)

## 1. 라이브러리 설치 및 설정

In [1]:
!pip install pandas numpy requests python-dotenv thinqconnect aiohttp -q


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import requests
import asyncio
import uuid
import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
from aiohttp import ClientSession
from thinqconnect.thinq_api import ThinQApi

load_dotenv(override=True)

# 센서 데이터 경로
SENSOR_BASE = Path('./sensor')  # 실제 경로로 수정

# Ollama
OLLAMA_URL   = f"http://{os.getenv('OLLAMA_HOST')}:{os.getenv('OLLAMA_PORT')}/api/generate"
OLLAMA_MODEL = os.getenv('OLLAMA_MODEL')

# ThinQ
THINQ_PAT       = os.getenv('THINQ_PAT')
THINQ_CLIENT_ID = str(uuid.uuid4())
THINQ_DEVICE_ID = os.getenv('THINQ_DEVICE_ID')

print('SENSOR_BASE:', SENSOR_BASE, '존재:', SENSOR_BASE.exists())
print('Ollama:', OLLAMA_URL)
print('ThinQ Device:', THINQ_DEVICE_ID)
print('설정 완료')

SENSOR_BASE: sensor 존재: True
Ollama: http://100.79.44.109:11434/api/generate
ThinQ Device: 31bc4282383642b47a47e313fa51f8211a3c4d8b8c319a76b87b3fc814656aef
설정 완료


## 2. 임계값 정의
- **classroom**: 학교보건법 시행규칙 제3조 별표2
- **hall**: 실내공기질관리법 시행규칙 별표2 (다중이용시설)
- **lab**: 학교보건법 준용

In [3]:
THRESHOLDS = {
    'classroom': {
        'co2':     {'comfortable': 700,  'normal': 1000},
        'aerosol': {'comfortable': 15,   'normal': 35},
        'temp':    {'min': 18, 'max': 28},
        'hum':     {'min': 30, 'max': 80},
    },
    'hall': {
        'co2':     {'comfortable': 700,  'normal': 1000},
        'aerosol': {'comfortable': 25,   'normal': 50},
        'temp':    None,
        'hum':     None,
    },
    'lab': {
        'co2':     {'comfortable': 700,  'normal': 1000},
        'aerosol': {'comfortable': 15,   'normal': 35},
        'temp':    {'min': 18, 'max': 28},
        'hum':     {'min': 30, 'max': 80},
    },
}

pd.DataFrame([
    ['classroom', '700/1000ppm', '15/35μg/m³', '18~28°C', '30~80%', '학교보건법 시행규칙'],
    ['hall',      '700/1000ppm', '25/50μg/m³', '-',       '-',      '실내공기질관리법'],
    ['lab',       '700/1000ppm', '15/35μg/m³', '18~28°C', '30~80%', '학교보건법 준용'],
], columns=['공간타입', 'CO2', 'PM2.5', '온도', '습도', '근거법령'])

,공간타입,CO2,PM2.5,온도,습도,근거법령
0,classroom,700/1000ppm,15/35μg/m³,18~28°C,30~80%,학교보건법 시행규칙
1,hall,700/1000ppm,25/50μg/m³,-,-,실내공기질관리법
2,lab,700/1000ppm,15/35μg/m³,18~28°C,30~80%,학교보건법 준용


## 3. 공간-센서 매핑

In [4]:
import pymysql
import pymysql.cursors

def get_conn():
    return pymysql.connect(
        host=os.getenv('DB_HOST'),
        user=os.getenv('DB_USER', 'kbu'),
        password=os.getenv('DB_PASSWORD', ''),
        db=os.getenv('DB_NAME', 'kbu_sensor'),
        port=int(os.getenv('DB_PORT', 3306)),
        charset='utf8mb4',
        cursorclass=pymysql.cursors.DictCursor,
    )

def load_room_mapping():
    """DB의 rooms + room_sensors 테이블에서 동적으로 매핑 로드"""
    conn = get_conn()
    mapping = {}
    try:
        with conn.cursor() as cur:
            cur.execute("SELECT room_id, name, floor, type FROM rooms")
            rooms = cur.fetchall()
            cur.execute("SELECT room_id, sensor_type, device_id FROM room_sensors ORDER BY room_id, sensor_type, device_id")
            sensors = cur.fetchall()

        sensor_map = {}
        for s in sensors:
            rid = s['room_id']
            if rid not in sensor_map:
                sensor_map[rid] = {'EDC': [], 'IRC': [], 'RDC': []}
            stype = s['sensor_type'].upper()
            if stype in sensor_map[rid]:
                sensor_map[rid][stype].append(s['device_id'])

        for r in rooms:
            rid = r['room_id']
            s = sensor_map.get(rid, {'EDC': [], 'IRC': [], 'RDC': []})
            mapping[rid] = {
                'floor': r['floor'], 'type': r['type'], 'name': r['name'],
                'EDC': s['EDC'], 'IRC': s['IRC'], 'RDC': s['RDC'],
                'AC':  THINQ_DEVICE_ID if rid == '3F-LEFT' else None,
            }
    finally:
        conn.close()
    return mapping

ROOM_MAPPING = load_room_mapping()

print('공간 매핑 완료 (DB 동적 로드)')
pd.DataFrame([
    [k, v['type'], len(v['EDC']), len(v['IRC']), len(v['RDC']), '✅' if v['AC'] else '-']
    for k, v in ROOM_MAPPING.items()
], columns=['공간', '타입', 'EDC', 'IRC', 'RDC', 'AC'])


공간 매핑 완료 (DB 동적 로드)


,공간,타입,EDC,IRC,RDC,AC
0,2F-HALL,hall,1,0,0,-
1,2F-LEFT,classroom,4,4,4,-
2,2F-RIGHT,lab,6,1,1,-
3,3F-HALL,hall,1,0,0,-
4,3F-LEFT,classroom,4,5,5,✅
5,3F-RIGHT,lab,6,1,1,-


## 4. CSV 읽기 함수

In [5]:
def get_latest_csv(subdir: str, device_id: str) -> Path | None:
    device_dir = SENSOR_BASE / subdir / device_id
    if not device_dir.exists():
        return None
    csvs = sorted(device_dir.glob('*.csv'))
    return csvs[-1] if csvs else None

def read_latest_row(subdir: str, device_id: str) -> pd.Series | None:
    path = get_latest_csv(subdir, device_id)
    if path is None:
        return None
    try:
        df = pd.read_csv(path)
        return df.iloc[-1] if not df.empty else None
    except Exception as e:
        print(f'read error {path}: {e}')
        return None

def read_recent_agg(device_id: str, hours: int = 3) -> pd.DataFrame:
    """최근 N시간 agg 데이터 (1h period만 사용)"""
    path = get_latest_csv('telemetry-agg', device_id)
    if path is None:
        return pd.DataFrame()
    try:
        df = pd.read_csv(path)
        df = df[df['period'] == '1h'].copy()
        df['createdAt'] = pd.to_datetime(df['createdAt'])
        return df.sort_values('createdAt').tail(hours)
    except:
        return pd.DataFrame()

print('CSV 읽기 함수 정의 완료')

CSV 읽기 함수 정의 완료


## 5. 온톨로지 변환
- `Room → hasSensor → {EDC, IRC, RDC}`
- `Sensor → hasMeasurement → Measurement`
- `Room → hasOccupancy` (RDC 주판단, IRC 보조)
- `Room → hasLocation` (RDC t1~t3 위치)
- `Room → hasTrend` (agg 트렌드)
- `Room → hasState` (판단 결과)

In [6]:
def build_ontology(room_id: str, mapping: dict) -> dict:
    edc_list, rdc_list, irc_list = [], [], []

    for device_id in mapping['EDC']:
        row = read_latest_row('telemetry-cal', device_id)
        if row is not None:
            edc_list.append({
                'device_id': device_id,
                'co2': row.get('co2'), 'aerosol': row.get('aerosol'),
                'gas': row.get('gas'), 'temp': row.get('temp'), 'hum': row.get('hum'),
            })

    for device_id in mapping['RDC']:
        row = read_latest_row('radar-frame', device_id)
        if row is not None:
            rdc_list.append({
                'device_id':    device_id,
                'target_count': int(row.get('targetCount', 0)),
                't1': {'x': row.get('t1_x'), 'y': row.get('t1_y'), 'dist': row.get('t1_dist'), 'speed': row.get('t1_speed')},
                't2': {'x': row.get('t2_x'), 'y': row.get('t2_y'), 'dist': row.get('t2_dist'), 'speed': row.get('t2_speed')},
                't3': {'x': row.get('t3_x'), 'y': row.get('t3_y'), 'dist': row.get('t3_dist'), 'speed': row.get('t3_speed')},
            })

    for device_id in mapping['IRC']:
        row = read_latest_row('thermal-frame', device_id)
        if row is not None:
            irc_list.append({
                'device_id':    device_id,
                'stats_min':    row.get('stats_min'),
                'stats_max':    row.get('stats_max'),
                'stats_avg':    row.get('stats_avg'),
                'stats_center': row.get('stats_center'),
            })

    def avg(lst, key):
        vals = [x[key] for x in lst if x.get(key) is not None and not pd.isna(x[key])]
        return round(sum(vals)/len(vals), 2) if vals else None

    def wmax(lst, key):
        vals = [x[key] for x in lst if x.get(key) is not None and not pd.isna(x[key])]
        return max(vals) if vals else None

    total_count = sum(r['target_count'] for r in rdc_list)
    trend_df    = read_recent_agg(mapping['EDC'][0]) if mapping['EDC'] else pd.DataFrame()

    return {
        'room_id': room_id, 'floor': mapping['floor'], 'type': mapping['type'],
        'hasSensor': {'EDC': [s['device_id'] for s in edc_list],
                      'RDC': [s['device_id'] for s in rdc_list],
                      'IRC': [s['device_id'] for s in irc_list]},
        'hasMeasurement': {
            'co2': avg(edc_list,'co2'), 'aerosol': avg(edc_list,'aerosol'),
            'gas': avg(edc_list,'gas'), 'temp': avg(edc_list,'temp'), 'hum': avg(edc_list,'hum'),
            'stats_min': avg(irc_list,'stats_min'), 'stats_max': wmax(irc_list,'stats_max'),
            'stats_avg': avg(irc_list,'stats_avg'), 'stats_center': avg(irc_list,'stats_center'),
        },
        'hasOccupancy':  total_count > 0 if rdc_list else None,
        'occupantCount': total_count,
        'hasLocation':   rdc_list,
        'hasTrend':      trend_df,
        'hasState':      None,
        'reason':        [],
        'prediction':    None,
    }

print('온톨로지 변환 함수 정의 완료')

온톨로지 변환 함수 정의 완료


## 6. 트렌드 예측 (agg 1h 기반)

In [7]:
def predict_co2(current_co2: float, trend_df: pd.DataFrame, threshold: float) -> dict:
    if trend_df.empty or len(trend_df) < 2:
        return {'trend': 'unknown', 'change_rate_per_h': None, 'predicted_co2': None, 'minutes_to_danger': None}
    co2_col = 'co2_avg' if 'co2_avg' in trend_df.columns else trend_df.columns[4]
    vals = trend_df[co2_col].values
    change_rate = (vals[-1] - vals[0]) / len(vals)
    predicted   = current_co2 + change_rate
    minutes_to_danger = round(((threshold - current_co2) / change_rate) * 60) if change_rate > 0 and current_co2 < threshold else None
    trend = 'increasing' if change_rate > 10 else 'decreasing' if change_rate < -10 else 'stable'
    return {
        'trend': trend,
        'change_rate_per_h': round(change_rate, 1),
        'predicted_co2': round(predicted, 1),
        'minutes_to_danger': minutes_to_danger,
    }

print('트렌드 예측 함수 정의 완료')

트렌드 예측 함수 정의 완료


## 7. 규칙 판단 엔진
판단 순서: CO2 → PM2.5 → 온도 → 습도 → IRC 보조 → 복합 → 재실 → 트렌드

In [8]:
def rule_engine(ontology: dict) -> dict:
    m         = ontology['hasMeasurement']
    occupied  = ontology['hasOccupancy']
    th        = THRESHOLDS[ontology['type']]
    trend_df  = ontology['hasTrend']
    reason    = []
    status    = 'comfortable'
    anomaly_count = 0

    co2 = m.get('co2'); aerosol = m.get('aerosol')
    temp = m.get('temp'); hum = m.get('hum')
    stats_max = m.get('stats_max'); stats_center = m.get('stats_center')

    if co2 is None and aerosol is None:
        ontology['hasState'] = 'abnormal'
        ontology['reason']   = ['센서 데이터 없음']
        return ontology

    # CO2
    if co2 is not None:
        if co2 > th['co2']['normal']:
            status = 'danger'; anomaly_count += 1
            reason.append(f'CO2 위험: {co2}ppm (기준 > {th["co2"]["normal"]})')
        elif co2 > th['co2']['comfortable']:
            if status != 'danger': status = 'normal'
            anomaly_count += 1
            reason.append(f'CO2 보통: {co2}ppm (기준 {th["co2"]["comfortable"]}~{th["co2"]["normal"]})')

    # PM2.5
    if aerosol is not None:
        if aerosol > th['aerosol']['normal']:
            status = 'danger'; anomaly_count += 1
            reason.append(f'PM2.5 위험: {aerosol}μg/m³ (기준 > {th["aerosol"]["normal"]})')
        elif aerosol > th['aerosol']['comfortable']:
            if status != 'danger': status = 'normal'
            anomaly_count += 1
            reason.append(f'PM2.5 보통: {aerosol}μg/m³')

    # 온도
    if temp is not None and th['temp'] is not None:
        if not (th['temp']['min'] <= temp <= th['temp']['max']):
            if status != 'danger': status = 'normal'
            anomaly_count += 1
            reason.append(f'온도 이상: {temp}°C (기준 {th["temp"]["min"]}~{th["temp"]["max"]})')

    # 습도
    if hum is not None and th['hum'] is not None:
        if not (th['hum']['min'] <= hum <= th['hum']['max']):
            if status != 'danger': status = 'normal'
            anomaly_count += 1
            reason.append(f'습도 이상: {hum}% (기준 {th["hum"]["min"]}~{th["hum"]["max"]})')

    # IRC 보조 판단
    if stats_max is not None:
        if stats_max >= 50:
            status = 'danger'
            reason.append(f'열화상 과열 감지: {stats_max}°C')
        elif stats_center is not None and stats_center >= 35 and occupied is None:
            occupied = True
            reason.append(f'IRC 재실 보조 감지: center {stats_center}°C')

    # 복합 판단 (2개 이상 동시 이상 → 위험도 상향)
    if anomaly_count >= 2 and status == 'normal':
        status = 'danger'
        reason.append(f'복합 이상: {anomaly_count}개 항목 동시 초과 → 위험도 상향')

    # 재실 기반 최종 판단 (논문 4.2절)
    if occupied is True and status == 'danger':
        reason.append('재실 상태 위험 → 즉각 조치 필요')
    elif occupied is False and status == 'danger':
        status = 'abnormal'
        reason.append('무재실 위험 → 비정상 (원인 불명)')
    elif occupied is None:
        reason.append('재실 정보 없음 (레이더 미설치)')

    # 트렌드 예측
    prediction = None
    if co2 is not None and not trend_df.empty:
        pred = predict_co2(co2, trend_df, th['co2']['normal'])
        prediction = pred
        if pred['minutes_to_danger']:
            reason.append(f'CO2 증가 추세: 약 {pred["minutes_to_danger"]}분 후 기준치 초과 예상')

    if status == 'comfortable':
        reason.append('모든 환경 기준 정상')

    ontology['hasState']   = status
    ontology['reason']     = reason
    ontology['prediction'] = prediction
    ontology['hasOccupancy'] = occupied
    return ontology

print('규칙 판단 함수 정의 완료')

규칙 판단 함수 정의 완료


## 8. LLM 해석 (Ollama)

In [9]:
def llm_explain(ontology: dict) -> str:
    m        = ontology['hasMeasurement']
    status   = ontology['hasState']
    occupied = ontology['hasOccupancy']
    count    = ontology['occupantCount']
    pred     = ontology['prediction']
    reason   = ' / '.join(ontology['reason'])
    status_kr = {'comfortable':'쾌적','normal':'보통','danger':'위험','abnormal':'비정상'}.get(status, status)

    trend_info = f"\n- CO2 증가 추세: 약 {pred['minutes_to_danger']}분 후 기준치 초과 예상" if pred and pred.get('minutes_to_danger') else ''

    prompt = f"""당신은 실내 환경 모니터링 시스템입니다.
아래 센서 데이터와 판단 결과만을 기반으로 설명을 생성하세요.
절대 데이터에 없는 내용을 추측하거나 추가하지 마세요.

[공간 정보]
- 공간: {ontology['room_id']} ({ontology['type']})
- 재실 인원: {count}명

[센서 데이터]
- CO2: {m.get('co2')}ppm
- PM2.5: {m.get('aerosol')}μg/m³
- 온도: {m.get('temp')}°C
- 습도: {m.get('hum')}%
- 열화상 최고: {m.get('stats_max')}°C{trend_info}

[판단 결과]
- 상태: {status_kr}
- 판단 근거: {reason}

위 정보만을 바탕으로 다음 두 가지를 작성하세요:
1. 현재 환경 상태 설명 (2문장 이내)
2. 행동 지침 (1~2가지)"""

    try:
        response = requests.post(OLLAMA_URL, json={
            'model': OLLAMA_MODEL, 'prompt': prompt, 'stream': False,
        }, timeout=600)
        return response.json().get('response', '').strip()
    except Exception as e:
        return f'LLM 오류: {e}'

print('LLM 함수 정의 완료')

LLM 함수 정의 완료


## 9. 에어컨 제어 (ThinQ Connect API)
### 제어 로직
- `danger` + 재실 있음 → AC ON (온도 기반 목표온도 설정)
- `comfortable/normal` + 재실 없음 → AC OFF
- `abnormal` → AC OFF + 경고
- 그 외 → 제어 없음

In [10]:
async def ac_control(device_id: str, power: str, target_temp: int = 24):
    """에어컨 제어 (POWER_ON / POWER_OFF)"""
    async with ClientSession() as session:
        api = ThinQApi(
            session=session,
            access_token=THINQ_PAT,
            country_code='KR',
            client_id=THINQ_CLIENT_ID
        )
        try:
            await api.async_post_device_control(
                device_id=device_id,
                payload={'operation': {'airConOperationMode': power}}
            )
            if power == 'POWER_ON':
                await api.async_post_device_control(
                    device_id=device_id,
                    payload={'temperature': {'targetTemperature': target_temp}}
                )
            print(f'  → AC {power} 완료 (목표온도: {target_temp}°C)')
        except Exception as e:
            print(f'  → AC 제어 스킵: {e}')
            
async def apply_ac_control(result: dict, mapping: dict):
    """판단 결과 → 에어컨 제어
    - 위험 상태 → AC ON
    - 위험 아님 → AC OFF
    """
    device_id = mapping.get('AC')
    room_id   = result['room_id']
    state     = result['hasState']
    temp      = result['hasMeasurement'].get('temp')

    if not device_id:
        print(f'[{room_id}] AC 미설치 → 제어 없음')
        return

    target_temp = 22 if temp and temp > 28 else 24

    if state == 'danger':
        print(f'[{room_id}] 위험 상태 → AC ON')
        await ac_control(device_id, 'POWER_ON', target_temp)
    else:
        print(f'[{room_id}] {state} 상태 → AC OFF')
        await ac_control(device_id, 'POWER_OFF')

print('에어컨 제어 함수 정의 완료')

에어컨 제어 함수 정의 완료


## 10. 전체 실행 (판단 + LLM + 에어컨 제어)

In [11]:
STATUS_ICON = {'comfortable':'🟢','normal':'🟡','danger':'🔴','abnormal':'⚫',None:'⬜'}

async def run_all():
    results = []
    for room_id, mapping in ROOM_MAPPING.items():
        print(f'\n── {room_id} ──')
        ontology = build_ontology(room_id, mapping)
        result   = rule_engine(ontology)
        results.append(result)

        icon = STATUS_ICON.get(result['hasState'], '?')
        print(f'상태: {icon} {result["hasState"]} | 재실: {result["hasOccupancy"]} ({result["occupantCount"]}명)')
        for r in result['reason']:
            print(f'  - {r}')

        await apply_ac_control(result, mapping)

    print('\n전체 판단 완료')
    return results

results = await run_all()


── 2F-HALL ──
상태: 🟡 normal | 재실: None (0명)
  - CO2 보통: 814.0ppm (기준 700~1000)
  - 재실 정보 없음 (레이더 미설치)
[2F-HALL] AC 미설치 → 제어 없음

── 2F-LEFT ──
상태: 🟢 comfortable | 재실: False (0명)
  - 모든 환경 기준 정상
[2F-LEFT] AC 미설치 → 제어 없음

── 2F-RIGHT ──
상태: 🟡 normal | 재실: False (0명)
  - CO2 보통: 808.33ppm (기준 700~1000)
[2F-RIGHT] AC 미설치 → 제어 없음

── 3F-HALL ──
상태: 🟡 normal | 재실: None (0명)
  - CO2 보통: 706.0ppm (기준 700~1000)
  - 재실 정보 없음 (레이더 미설치)
[3F-HALL] AC 미설치 → 제어 없음

── 3F-LEFT ──
상태: 🟢 comfortable | 재실: False (0명)
  - 모든 환경 기준 정상
[3F-LEFT] comfortable 상태 → AC OFF
  → AC 제어 스킵: ThinQAPIException: COMMAND_NOT_SUPPORTED_IN_POWER_OFF (2304) - Command not supported in POWER OFF

── 3F-RIGHT ──
상태: ⚫ abnormal | 재실: False (0명)
  - CO2 위험: 1728.0ppm (기준 > 1000)
  - 무재실 위험 → 비정상 (원인 불명)
[3F-RIGHT] AC 미설치 → 제어 없음

전체 판단 완료


## 11. 결과 시각화

In [12]:
rows = []
for r in results:
    m    = r['hasMeasurement']
    pred = r['prediction']
    rows.append({
        '공간':       r['room_id'],
        '타입':       r['type'],
        '상태':       STATUS_ICON.get(r['hasState'],'?') + ' ' + (r['hasState'] or '-'),
        '재실인원':   r['occupantCount'],
        'CO2':        m.get('co2'),
        'PM2.5':      m.get('aerosol'),
        '온도':       m.get('temp'),
        '습도':       m.get('hum'),
        'IRC_max':    m.get('stats_max'),
        'CO2_추세':   pred['trend'] if pred else '-',
        '위험예측(분)': pred['minutes_to_danger'] if pred else '-',
    })

pd.DataFrame(rows)

,공간,타입,상태,재실인원,CO2,PM2.5,온도,습도,IRC_max,CO2_추세,위험예측(분)
0,2F-HALL,hall,🟡 normal,0,814.00,5.0,25.80,31.50,NaN,decreasing,None
1,2F-LEFT,classroom,🟢 comfortable,0,586.00,5.0,24.42,36.50,24.6,stable,None
2,2F-RIGHT,lab,🟡 normal,0,808.33,5.0,23.67,34.12,19.7,decreasing,None
3,3F-HALL,hall,🟡 normal,0,706.00,5.0,26.80,31.50,NaN,decreasing,None
4,3F-LEFT,classroom,🟢 comfortable,0,535.75,5.0,26.15,32.60,35.1,stable,None
5,3F-RIGHT,lab,⚫ abnormal,0,1728.00,5.0,21.78,32.88,34.9,decreasing,None


## 12. LLM 해석 실행

In [13]:
for r in results:
    print(f"\n=== {r['room_id']} ({r['hasState']}) ===")
    explanation = llm_explain(r)
    print(explanation)
    r['llm_description'] = explanation


=== 2F-HALL (normal) ===
**1. 현재 환경 상태 설명**
현재 2F-HALL의 환경 상태는 보통이며, CO2 농도는 814.0ppm으로 기준 범위 내에 있습니다. 온도는 25.8°C, 습도는 31.5%로 측정되었습니다.

**2. 행동 지침**
*   현재 환경 상태가 보통이므로 별도의 환경 조치(환기/냉난방 등)는 필요하지 않습니다.
*   재실 인원 정보가 없는 상태이므로, 인원 변화에 따른 환경 모니터링에 유의해야 합니다.

=== 2F-LEFT (comfortable) ===
**1. 현재 환경 상태 설명**
현재 2F-LEFT 강의실은 재실 인원 없이 모든 환경 기준이 정상인 쾌적한 상태입니다. 측정된 CO2(586.0ppm), PM2.5(5.0μg/m³), 온도(24.42°C), 습도(36.5%) 등 모든 항목이 정상 범위 내에 있습니다.

**2. 행동 지침**
현재의 쾌적한 환경 상태를 유지해 주십시오.

=== 2F-RIGHT (normal) ===
**1. 현재 환경 상태 설명**
2F-RIGHT (lab) 공간은 현재 재실 인원이 없으며, 전반적인 환경 상태는 보통으로 판단됩니다. CO2 농도는 808.33ppm으로 기준 범위(700~1000ppm) 내에 있어 정상적인 수준을 유지하고 있습니다.

**2. 행동 지침**
*   현재 환경 상태가 보통이므로 별도의 환경 조치는 필요하지 않습니다.
*   지속적인 환경 모니터링을 유지해 주십시오.

=== 3F-HALL (normal) ===
**1. 현재 환경 상태 설명**
현재 3F-HALL의 환경 상태는 보통이며, CO2 농도는 706.0ppm으로 기준 범위 내에 있습니다. 온도는 26.8°C, 습도는 31.5%로 측정되었습니다.

**2. 행동 지침**
현재 환경 상태가 보통이므로 추가적인 조치는 필요하지 않습니다.

=== 3F-LEFT (comfortable) ===
**1. 현재 환경 상태 설명**
3F-LEFT 교실의 현재 환경 상태는 쾌적하며, 모든 환경 

## 다음 단계
- **DB 스키마 확정**: 검증된 컬럼으로 MariaDB 재설계 (radar t1~t3, thermal stats_min/center, telemetry_agg 추가)
- **watchdog 통합**: 5분마다 자동 실행
- **에어컨 확장**: AC_MAP에 나머지 공간 device_id 추가
- **대시보드**: 도면 레이아웃 기반 시각화